# Supervised Fine-Tuning (SFT) with Serverless customization on SageMaker AI

## Lab 1.2b - Import Pre-trained Model from S3

This notebook is an **alternative to notebook 2 (fine-tuning)**. Instead of running a full SFT training job (which can take a long time), you can use a pre-trained model that is already available on S3.

This notebook will:
1. Import the model artifacts from S3
2. Create a Model Package Group (if it doesn't exist)
3. Register the model as a Model Package in the SageMaker Model Registry

After running this notebook, you can proceed directly to **notebook 3 (evaluation)** and **notebook 4 (deployment)**.

---

## Prerequisites

### Install requirements

In [ ]:
# No extra packages needed — boto3 handles the download.


### AWS Access Setup and dependencies

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix
region = sess.boto_region_name

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {region}")

from config import BASE_MODEL_ID

base_model_id = BASE_MODEL_ID
base_model_jumpstart_id = base_model_id
base_model_shortname = "qwen3-4b"


## Download pre-trained model and upload to your S3 bucket

Download the model artifacts from the workshop asset URL and upload them to your own S3 bucket.

In [ ]:
# S3 destination for the pre-trained model artifacts.
# Matches the training output path used by notebook 2 (<base_model_id>-contractnli).
base_project_path = (f"s3://{bucket_name}/{default_prefix}/{base_model_id}-contractnli/" if default_prefix else f"s3://{bucket_name}/{base_model_id}-contractnli/")

# Download the pre-trained (already fine-tuned) merged checkpoint that ships with
# this workshop and upload it to your own S3 bucket. The checkpoint is the HF-format
# `hf_merged/` model produced by the training notebook; it lives in the same
# workshop asset bundle as the pre-computed evaluation results.
import os

import boto3

# S3 destination in *your* bucket. Matches the training output path used by the
# fine-tuning notebook so the evaluation/deployment notebooks resolve it the same way.
model_s3_upload_uri = f"{base_project_path}checkpoints/hf_merged/"
pretrained_model_s3_uri = base_project_path

# Workshop asset location (same bucket/prefix scheme as the pre-computed eval results).
PRECOMPUTED_BUCKET = "ws-assets-prod-iad-r-iad-ed304a55c2ca1aee"
CHECKPOINT_PREFIX = "548b5be9-2da8-4c93-82f7-b0b474108ab3/lab1/checkpoints/hf_merged"

_src = boto3.client("s3", region_name="us-east-1")
_dst = boto3.client("s3")

# Parse the destination bucket/prefix from the target URI.
_dst_parts = model_s3_upload_uri.replace("s3://", "").split("/", 1)
_dst_bucket = _dst_parts[0]
_dst_prefix = _dst_parts[1] if len(_dst_parts) > 1 else ""

# Copy every checkpoint object from the workshop asset bucket into your bucket.
_paginator = _src.get_paginator("list_objects_v2")
_count = 0
for _page in _paginator.paginate(Bucket=PRECOMPUTED_BUCKET, Prefix=CHECKPOINT_PREFIX):
    for _obj in _page.get("Contents", []):
        _key = _obj["Key"]
        _rel = _key[len(CHECKPOINT_PREFIX):].lstrip("/")
        if not _rel:
            continue
        _body = _src.get_object(Bucket=PRECOMPUTED_BUCKET, Key=_key)["Body"]
        _dst.upload_fileobj(_body, _dst_bucket, _dst_prefix + _rel)
        _count += 1

print(f"Copied {_count} checkpoint file(s) to {model_s3_upload_uri}")
print(f"\nPre-trained model S3 URI: {pretrained_model_s3_uri}")

### Verify the model artifacts exist on S3

In [ ]:
s3_client = boto3.client("s3")

# Parse bucket and prefix from the S3 URI
s3_parts = pretrained_model_s3_uri.replace("s3://", "").split("/", 1)
s3_bucket = s3_parts[0]
s3_prefix = s3_parts[1] if len(s3_parts) > 1 else ""

response = s3_client.list_objects_v2(Bucket=s3_bucket, Prefix=s3_prefix, MaxKeys=10)

if "Contents" in response:
    print(f"✅ Found {response['KeyCount']} objects (showing up to 10):")
    for obj in response["Contents"]:
        print(f"  - {obj['Key']} ({obj['Size']:,} bytes)")
else:
    print("❌ No objects found at the specified S3 path. Please verify the URI.")

---

## Create Model Package Group

Create or retrieve the Model Package Group in the SageMaker Model Registry. This is the same group used by the fine-tuning notebook, so the evaluation and deployment notebooks will work seamlessly.

In [ ]:
import hashlib
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup

from config import BASE_MODEL_ID
base_model_id = BASE_MODEL_ID

MAX_MPG_NAME_LENGTH = 63
suffix = "-contractnli-sft-mpg"

candidate = f"{base_model_id}{suffix}"
if len(candidate) > MAX_MPG_NAME_LENGTH:
    digest = hashlib.sha1(base_model_id.encode()).hexdigest()[:6]
    # reserve room for the suffix, a hyphen separator, and the 6-char hash
    keep = MAX_MPG_NAME_LENGTH - len(suffix) - len(digest) - 1
    truncated = base_model_id[:keep].rstrip("-")
    model_package_group_name = f"{truncated}-{digest}{suffix}"
else:
    model_package_group_name = candidate

try:
    model_package_group = ModelPackageGroup.get(
        model_package_group_name=model_package_group_name
    )
    print(f"Model Package Group already exists: {model_package_group_name}")
except ClientError:
    model_package_group = ModelPackageGroup.create(
        model_package_group_name=model_package_group_name,
        model_package_group_description="ContractNLI NDA checklist review — serverless SFT",
    )
    print(f"Created Model Package Group: {model_package_group_name}")

---

## Register the Model Package

Register the pre-trained model from S3 as a versioned Model Package. This makes it available for the evaluation and deployment notebooks, just as if it had been produced by a training job.

In [ ]:
sm_client = boto3.client("sagemaker", region_name=region)


response = sm_client.create_model_package(
    ModelPackageGroupName=model_package_group_name,
    ModelPackageDescription="Pre-trained SFT model imported from S3",
    InferenceSpecification={
        "Containers": [
            {
                "ModelDataSource": {
                    "S3DataSource": {
                        "S3Uri": pretrained_model_s3_uri,
                        "S3DataType": "S3Prefix",
                        "CompressionType": "None",
                    }
                },
                "BaseModel": {
                    "HubContentName": "huggingface-reasoning-qwen3-4b",
                    "HubContentVersion": "1.39.0",
                    "RecipeName": "llmft_qwen3_4b_seq4k_gpu_sft_lora",
                },
            }
        ],
    },
    ModelApprovalStatus="Approved",
)

model_package_arn = response["ModelPackageArn"]
print(f"✅ Created Model Package: {model_package_arn}")

### Verify the registered Model Package

In [ ]:
from sagemaker.core.resources import ModelPackage

model_package = ModelPackage.get(model_package_arn)

print(f"Model Package ARN: {model_package_arn}")
print(f"Model Package Group: {model_package_group_name}")
print(f"Status: {model_package.model_approval_status}")
print(f"S3 URI: {model_package.inference_specification.containers[0].model_data_source.s3_data_source.s3_uri}")
print(f"\n✅ Model is registered and ready. You can now proceed to:")
print(f"   - Notebook 3 (evaluation): 3-evaluation.ipynb")
print(f"   - Notebook 4 (deployment): 4-deployment.ipynb")